# Embalses y generación eléctrica en Colombia durante El Niño
## 01 · Extracción y limpieza de datos de XM

**Fuente:** API pública de XM (operador del mercado eléctrico colombiano) a través de la librería `pydataxm`.
**Periodo:** 2015-01-01 a 2026-10-04.
**Objetivo:** dejar listas las tablas que alimentan el modelo de Power BI para comparar los episodios de El Niño 2015-16, 2023-24 y 2026.

**Cómo ejecutarlo:** en un runtime nuevo de Colab, con *Entorno de ejecución → Ejecutar todo*. Las descargas por bloques de 31 días tardan varios minutos. Al final, la última celda valida los conteos contra los de la corrida original y descarga un `.zip` con los CSV.

**Salidas:**
- `data/processed/`: las 5 tablas del modelo.
- `data/reference/`: series y catálogos de apoyo.

In [ ]:
!pip install pydataxm -q

import os, time, glob, shutil
import pandas as pd
from pydataxm import pydataxm

api = pydataxm.ReadDB()
pd.set_option('display.max_colwidth', None)

FECHA_INICIO = "2015-01-01"
FECHA_FIN = "2026-10-04"
DIAS_ESPERADOS = (pd.Timestamp(FECHA_FIN) - pd.Timestamp(FECHA_INICIO)).days + 1   # 4295

# Ventanas de comparación (duración de cada episodio de El Niño)
ventanas = {
    "nino_2015_16": ("2015-03-01", "2016-05-31"),
    "nino_2023_24": ("2023-06-01", "2024-04-30"),
    "nino_2026":    ("2026-06-01", "2026-10-04"),   # en curso
}

columnas_hora = [f"Values_Hour{str(i).zfill(2)}" for i in range(1, 25)]

## 1. Catálogo de métricas de XM
Se lista el catálogo completo y se buscan las métricas de interés para confirmar sus `MetricId` exactos antes de descargar nada.

In [ ]:
catalogo = api.get_collections()

palabras = ['Embalse', 'Volumen', 'Aportes', 'Capacidad', 'Generación', 'Precio Bolsa']

for palabra in palabras:
    resultado = catalogo[catalogo['MetricName'].str.contains(palabra, case=False, na=False)]
    print(f"\n--- {palabra} ---")
    print(resultado[['MetricId', 'MetricName', 'Entity']])

## 2. Función de descarga por bloques
La API acepta como máximo 31 días por consulta en granularidad diaria. La función parte el rango en bloques de 31 días, los descarga y los une. Imprime solo los errores y un resumen al final (con `verbose=True` imprime cada bloque).

In [ ]:
def descargar_serie_diaria(metric_id, entity, fecha_inicio, fecha_fin, verbose=False):
    """
    Descarga una métrica diaria de la API de XM partiendo el rango en
    bloques de 31 días (límite de la API para granularidad diaria).
    """
    rangos = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='31D')
    resultados = []

    for inicio in rangos:
        fin = min(inicio + pd.Timedelta(days=30), pd.Timestamp(fecha_fin))

        try:
            df = api.request_data(
                metric_id,
                entity,
                inicio.strftime('%Y-%m-%d'),
                fin.strftime('%Y-%m-%d')
            )
            resultados.append(df)
            if verbose:
                print(f"OK: {inicio.date()} a {fin.date()} — {len(df)} filas")
        except Exception as e:
            print(f"ERROR en {inicio.date()} a {fin.date()}: {e}")

        time.sleep(0.5)  # pausa para no saturar la API

    resultado = pd.concat(resultados, ignore_index=True)
    print(f"{metric_id} ({entity}): {len(resultados)} bloques, {len(resultado)} filas")
    return resultado

## 3. Embalses: listado, capacidad y ranking (top 15)
Se excluye `AGREGADO BOGOTA` del ranking individual por no ser un embalse individual. Con los 15 primeros se cubre el 95.72 % de la capacidad útil del país; el resto no entra al análisis detallado.

In [ ]:
listado_embalses = api.request_data("ListadoEmbalses", "Sistema", "2026-01-01", "2026-01-01")

capacidad_embalses = api.request_data("CapaUtilDiarEner", "Embalse", "2026-09-28", "2026-10-04")

# La capacidad no varía diario: nos quedamos con un solo día
capacidad_un_dia = capacidad_embalses[capacidad_embalses['Date'] == '2026-10-04'].copy()
capacidad_un_dia = capacidad_un_dia[capacidad_un_dia['Name'] != 'AGREGADO BOGOTA']

ranking = capacidad_un_dia.sort_values('Value', ascending=False).reset_index(drop=True)
ranking['Participacion_%'] = (ranking['Value'] / ranking['Value'].sum() * 100).round(2)
ranking['Acumulado_%'] = ranking['Participacion_%'].cumsum().round(2)

top15 = ['PENOL', 'GUAVIO', 'EL QUIMBO', 'TOPOCORO', 'CHUZA', 'ESMERALDA', 'RIOGRANDE2',
         'SALVAJINA', 'MIRAFLORES', 'SAN LORENZO', 'AMANI', 'CALIMA1', 'URRA1', 'PORCE II', 'BETANIA']

ranking

## 4. Series diarias del sistema (2015-2026)

**Hallazgo de calidad — filas duplicadas.** Al descargar por bloques aparecen filas repetidas (más filas de las que tiene el bloque, p. ej. jun-ago 2019 y ene-abr 2024). Son copias exactas (misma fecha, mismo valor) y se eliminan con `drop_duplicates`. La limpieza se valida contra el número exacto de días del rango.

In [ ]:
# 4.1 Volumen útil diario (%) — Sistema
volumen_historico = descargar_serie_diaria("PorcVoluUtilDiar", "Sistema", FECHA_INICIO, FECHA_FIN)

duplicados = volumen_historico[volumen_historico.duplicated(subset='Date', keep=False)]
print(f"Filas con fecha duplicada: {len(duplicados)}")

volumen_historico_limpio = volumen_historico.drop_duplicates(subset=['Date', 'Value']).reset_index(drop=True)

print(f"Filas antes: {len(volumen_historico)}")
print(f"Filas después: {len(volumen_historico_limpio)}")
print(f"Filas esperadas ({FECHA_INICIO} a {FECHA_FIN}): {DIAS_ESPERADOS}")
print(f"Fechas aún duplicadas: {volumen_historico_limpio['Date'].duplicated().sum()}")

In [ ]:
# 4.2 Aportes hídricos reales y media histórica — Sistema
aportes_reales = descargar_serie_diaria("AporEner", "Sistema", FECHA_INICIO, FECHA_FIN)
aportes_media_historica = descargar_serie_diaria("AporEnerMediHist", "Sistema", FECHA_INICIO, FECHA_FIN)

aportes_reales = aportes_reales.drop_duplicates(subset=['Date', 'Value']).reset_index(drop=True)
aportes_media_historica = aportes_media_historica.drop_duplicates(subset=['Date', 'Value']).reset_index(drop=True)

print(f"Aportes reales: {len(aportes_reales)} filas")
print(f"Media histórica: {len(aportes_media_historica)} filas")
print(f"Esperadas: {DIAS_ESPERADOS}")

In [ ]:
# 4.3 Precio de bolsa nacional (COP/kWh). Llega con 24 columnas de hora: se promedia el día (promedio simple)
info = catalogo[catalogo['MetricId'] == 'PrecBolsNaci'][['MetricName', 'Entity', 'MaxDays', 'MetricUnits']]
print(info.to_string())

precio_raw = descargar_serie_diaria("PrecBolsNaci", "Sistema", FECHA_INICIO, FECHA_FIN).drop_duplicates()
print("\nColumnas:", precio_raw.columns.tolist())

cols_h = [c for c in precio_raw.columns if c.startswith('Values_Hour')]
if cols_h:
    horas = precio_raw[cols_h].apply(pd.to_numeric, errors='coerce')
    print(f"Horas vacías o no numéricas: {horas.isna().sum().sum()}")
    precio = pd.DataFrame({'Date': precio_raw['Date'], 'Precio_Bolsa_Promedio': horas.mean(axis=1)})
else:
    precio = precio_raw[['Date', 'Value']].rename(columns={'Value': 'Precio_Bolsa_Promedio'})
    precio['Precio_Bolsa_Promedio'] = pd.to_numeric(precio['Precio_Bolsa_Promedio'], errors='coerce')

print(f"Fechas duplicadas antes de depurar: {precio['Date'].duplicated().sum()}")
precio = precio.drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)

print(f"Filas: {len(precio)} (esperado {DIAS_ESPERADOS}), nulos: {precio['Precio_Bolsa_Promedio'].isna().sum()}")
print(precio['Precio_Bolsa_Promedio'].describe().round(2).to_string())

print("\nPrecio máximo:")
print(precio.loc[precio['Precio_Bolsa_Promedio'].idxmax()].to_string())

## 5. Volumen útil (%) por embalse — top 15
Cada consulta devuelve todos los embalses; después se filtra al top 15. No todos tienen los 4,295 días: EL QUIMBO empieza el 2015-11-01 y MIRAFLORES tiene huecos (se revisan en la sección 8).

In [ ]:
vol_emb = descargar_serie_diaria("PorcVoluUtilDiar", "Embalse", FECHA_INICIO, FECHA_FIN).drop_duplicates()
print("\nColumnas:", vol_emb.columns.tolist())

vol_top = vol_emb[vol_emb['Name'].isin(top15)].copy()
vol_top['Value'] = pd.to_numeric(vol_top['Value'], errors='coerce')
print(f"\nEmbalses encontrados: {vol_top['Name'].nunique()} de 15")
print(f"Duplicados nombre+fecha: {vol_top.duplicated(subset=['Name', 'Date']).sum()}, nulos: {vol_top['Value'].isna().sum()}")

# Días por embalse (esperado 4295; menos si el embalse empezó a operar después de 2015)
print(vol_top.groupby('Name')['Date'].agg(['count', 'min', 'max']).to_string())

## 6. Generación por tipo de fuente

### 6.1 Catálogo de recursos
**Hallazgo de calidad — categorías inconsistentes.** En `Values_Type` aparecen `SOLAR` y `Solar` como categorías distintas. Se normaliza con `.str.upper()` antes de agrupar.

In [ ]:
listado_recursos = api.request_data("ListadoRecursos", "Sistema", "2026-01-01", "2026-01-01")

print("Values_Type tal como llega:")
print(listado_recursos['Values_Type'].value_counts())

recursos = listado_recursos[['Values_Code', 'Values_Name', 'Values_Type', 'Values_EnerSource']].copy()
recursos['Values_Type'] = recursos['Values_Type'].str.upper().str.strip()
recursos = recursos.drop_duplicates(subset='Values_Code')

print(f"\nRecursos en catálogo (únicos): {len(recursos)}")
print(recursos['Values_Type'].value_counts())

### 6.2 Hallazgo de calidad — las horas llegan como texto
La generación (`Gene`, por recurso, en kWh) trae 24 columnas de hora. Con una semana de prueba se comprueba que llegan como **texto** (`object`) y que las celdas que no se pueden convertir son siempre cadenas vacías, concentradas en las horas sin sol y en recursos solares. Se interpretan como horas sin generación (0 kWh).

Sumar las columnas sin convertirlas **concatena el texto** en vez de sumar; por eso se convierte con `pd.to_numeric` antes.

In [ ]:
prueba = descargar_serie_diaria("Gene", "Recurso", "2026-09-01", "2026-09-07").drop_duplicates()

print("Tipos de las columnas de hora (como llegan de la API):")
print(prueba[columnas_hora].dtypes.value_counts())

crudo = prueba[columnas_hora]
horas_num = crudo.apply(pd.to_numeric, errors='coerce')
mascara_falla = horas_num.isna() & crudo.notna()

print(f"\nCeldas que no se pudieron convertir: {mascara_falla.sum().sum()}")
print("Valores distintos que fallaron:")
print(crudo[mascara_falla].stack().map(repr).value_counts().head(10))

print("\nFallas por hora del día:")
print(mascara_falla.sum().to_string())

prueba['n_fallas'] = mascara_falla.sum(axis=1)
prueba['Tipo'] = prueba['Values_code'].map(recursos.set_index('Values_Code')['Values_Type'])
print("\nFallas promedio por fila según tipo de recurso:")
print(prueba.groupby('Tipo', dropna=False)['n_fallas'].agg(['count', 'mean']).to_string())

# Unidades de la métrica según el catálogo de XM
print()
print(catalogo[catalogo['MetricId'] == 'Gene'][['MetricName', 'Entity', 'MetricUnits']].to_string())

### 6.3 Generación diaria por recurso en las 3 ventanas de El Niño
**Decisión de alcance.** La generación por recurso del periodo completo pesaría unos 2 millones de filas, así que se acota a las tres ventanas de comparación. Las horas se convierten a número (vacío = 0 kWh) y se suman a un total diario por recurso.

Luego se mide cuánta generación queda sin clasificar porque el catálogo de recursos es el vigente al consultar y puede no incluir plantas ya retiradas.

In [ ]:
generacion_por_ventana = {}
for nombre, (inicio, fin) in ventanas.items():
    print(f"\n=== {nombre}: {inicio} a {fin} ===")
    df = descargar_serie_diaria("Gene", "Recurso", inicio, fin).drop_duplicates()
    horas = df[columnas_hora].apply(pd.to_numeric, errors='coerce').fillna(0)   # vacío = 0 kWh
    df['Generacion_Diaria_kWh'] = horas.sum(axis=1)
    res = df[['Values_code', 'Date', 'Generacion_Diaria_kWh']].copy()
    dup = res.duplicated(subset=['Values_code', 'Date']).sum()
    generacion_por_ventana[nombre] = res
    print(f"{nombre}: {len(res)} filas, {res['Values_code'].nunique()} recursos, duplicados código+fecha: {dup}")

# Diagnóstico: generación sin tipo en el catálogo
print("\n--- Generación sin clasificar ---")
for nombre, df in generacion_por_ventana.items():
    sin = set(df['Values_code']) - set(recursos['Values_Code'])
    pct = df[df['Values_code'].isin(sin)]['Generacion_Diaria_kWh'].sum() / df['Generacion_Diaria_kWh'].sum()
    print(f"{nombre}: {len(sin)} recursos sin match, {pct:.2%} de la generación")

In [ ]:
# 6.4 Generación diaria por tipo de fuente (los recursos sin match quedan como 'SIN CLASIFICAR')
tipo_map = recursos.set_index('Values_Code')['Values_Type']

partes = []
for nombre, df in generacion_por_ventana.items():
    d = df.copy()
    d['Tipo'] = d['Values_code'].map(tipo_map).fillna('SIN CLASIFICAR')
    d['Ventana'] = nombre
    partes.append(d)
gen = pd.concat(partes, ignore_index=True)

gen_tipo = gen.groupby(['Ventana', 'Date', 'Tipo'], as_index=False)['Generacion_Diaria_kWh'].sum()
gen_tipo['Generacion_GWh'] = gen_tipo['Generacion_Diaria_kWh'] / 1e6

# Validación 1: días por ventana vs. esperados
for nombre, (ini, fin) in ventanas.items():
    esperado = (pd.Timestamp(fin) - pd.Timestamp(ini)).days + 1
    real = gen_tipo[gen_tipo['Ventana'] == nombre]['Date'].nunique()
    print(f"{nombre}: {real} días (esperado {esperado})")

# Validación 2: participación por tipo en cada ventana (%)
resumen = gen_tipo.pivot_table(index='Ventana', columns='Tipo', values='Generacion_GWh', aggfunc='sum')
print("\nParticipación por tipo (%):")
print((resumen.div(resumen.sum(axis=1), axis=0) * 100).round(2).to_string())

## 7. Calidad: unidades y huecos
- **Unidades:** aportes y capacidad vienen en kWh. El volumen útil figura como `%` en el catálogo pero los valores son fracciones entre 0 y 1.
- **MIRAFLORES:** se ubican sus días faltantes. No se interpolan: quedan como vacíos y se documentan.

In [ ]:
# 7.1 Unidades según el catálogo de XM
ids = ['PorcVoluUtilDiar', 'AporEner', 'AporEnerMediHist', 'CapaUtilDiarEner']
print(catalogo[catalogo['MetricId'].isin(ids)][['MetricId', 'MetricName', 'Entity', 'MetricUnits']].to_string())

# 7.2 Días faltantes de MIRAFLORES, agrupados en tramos consecutivos
mir = vol_top[vol_top['Name'] == 'MIRAFLORES']['Date']
faltan = pd.date_range(FECHA_INICIO, FECHA_FIN).difference(mir)
print(f"\nMIRAFLORES: {len(faltan)} días faltantes")
if len(faltan) > 0:
    s = pd.Series(faltan)
    tramo = (s.diff() != pd.Timedelta(days=1)).cumsum()
    print(s.groupby(tramo).agg(['min', 'max', 'count']).to_string())

## 8. Tablas finales del modelo
- `fact_sistema_diario`: una fila por día con volumen, aportes (real y media histórica) y precio de bolsa.
- `dim_embalse`: top 15 con capacidad útil (kWh) y región hídrica.
- `dim_ventana`: las tres ventanas de El Niño.
- Además se exportan `volumen_util_por_embalse` y `generacion_diaria_por_tipo`.

In [ ]:
def preparar(df, col_origen, col_nueva):
    out = df[['Date', col_origen]].copy()
    out[col_origen] = pd.to_numeric(out[col_origen], errors='coerce')
    return out.rename(columns={col_origen: col_nueva})

fact_sistema = (
    preparar(volumen_historico_limpio, 'Value', 'Volumen_Util_Fraccion')
    .merge(preparar(aportes_reales, 'Value', 'Aportes_kWh'), on='Date', how='outer')
    .merge(preparar(aportes_media_historica, 'Value', 'Aportes_MediaHist_kWh'), on='Date', how='outer')
    .merge(preparar(precio, 'Precio_Bolsa_Promedio', 'Precio_Bolsa_COP_kWh'), on='Date', how='outer')
    .sort_values('Date').reset_index(drop=True)
)

print(f"Filas: {len(fact_sistema)} (esperado {DIAS_ESPERADOS}), fechas duplicadas: {fact_sistema['Date'].duplicated().sum()}")
print("Nulos por columna:")
print(fact_sistema.isna().sum().to_string())
print(fact_sistema.describe().round(3).to_string())

# dim_embalse: top 15 con capacidad y región hídrica
regiones = (listado_embalses.drop_duplicates(subset='Values_Code')[['Values_Name', 'Values_HydroRegion']]
            .rename(columns={'Values_Name': 'Name', 'Values_HydroRegion': 'Region_Hidrica'}))

dim_embalse = ranking[ranking['Name'].isin(top15)][['Name', 'Value']].rename(columns={'Value': 'Capacidad_Util'})
dim_embalse = dim_embalse.merge(regiones, on='Name', how='left')
print("\n", dim_embalse.to_string())
print("Sin región:", dim_embalse[dim_embalse['Region_Hidrica'].isna()]['Name'].tolist())

dim_ventana = pd.DataFrame([
    {'Ventana': 'nino_2015_16', 'Evento': 'El Niño 2015-16',         'Inicio': '2015-03-01', 'Fin': '2016-05-31'},
    {'Ventana': 'nino_2023_24', 'Evento': 'El Niño 2023-24',         'Inicio': '2023-06-01', 'Fin': '2024-04-30'},
    {'Ventana': 'nino_2026',    'Evento': 'El Niño 2026 (en curso)', 'Inicio': '2026-06-01', 'Fin': '2026-10-04'},
])

## 9. Exportar
Se guardan los CSV en `data/processed/` (tablas del modelo) y `data/reference/` (series y catálogos de apoyo), y se descarga todo en un `.zip`.

In [ ]:
from google.colab import files

os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/reference', exist_ok=True)

# Tablas del modelo
fact_sistema.to_csv('data/processed/fact_sistema_diario.csv', index=False)
vol_top[['Name', 'Date', 'Value']].to_csv('data/processed/volumen_util_por_embalse.csv', index=False)
gen_tipo.to_csv('data/processed/generacion_diaria_por_tipo.csv', index=False)
dim_embalse.to_csv('data/processed/dim_embalse.csv', index=False)
dim_ventana.to_csv('data/processed/dim_ventana.csv', index=False)

# Referencia
listado_recursos.to_csv('data/reference/listado_recursos.csv', index=False)
precio.to_csv('data/reference/precio_bolsa_diario.csv', index=False)
volumen_historico_limpio.to_csv('data/reference/volumen_historico_sistema.csv', index=False)
aportes_reales.to_csv('data/reference/aportes_reales.csv', index=False)
aportes_media_historica.to_csv('data/reference/aportes_media_historica.csv', index=False)

shutil.make_archive('datos_embalses_nino', 'zip', 'data')
files.download('datos_embalses_nino.zip')

## 10. Validación contra la corrida original
Los números esperados son los de la primera ejecución del pipeline. Si alguno dice `REVISAR`, hay que investigar antes de usar los datos. Una advertencia: los catálogos (`ListadoRecursos`, `ListadoEmbalses`) reflejan el día en que se consultan, así que los recursos sin clasificar pueden variar un poco en una ejecución posterior.

In [ ]:
def verificar(nombre, real, esperado):
    print(f"{'OK      ' if real == esperado else 'REVISAR '} {nombre}: {real} (esperado {esperado})")

verificar("Días de la serie del sistema", len(fact_sistema), 4295)
verificar("Fechas duplicadas en fact_sistema_diario", int(fact_sistema['Date'].duplicated().sum()), 0)
verificar("Nulos en fact_sistema_diario", int(fact_sistema.isna().sum().sum()), 0)
verificar("Embalses del top 15 encontrados", vol_top['Name'].nunique(), 15)
verificar("Días faltantes de MIRAFLORES", len(faltan), 341)

esperado_filas = {"nino_2015_16": 63144, "nino_2023_24": 82478, "nino_2026": 63390}
esperado_dias = {"nino_2015_16": 458, "nino_2023_24": 335, "nino_2026": 126}
for nombre in ventanas:
    verificar(f"Filas de generación {nombre}", len(generacion_por_ventana[nombre]), esperado_filas[nombre])
    verificar(f"Días de generación {nombre}", gen_tipo[gen_tipo['Ventana'] == nombre]['Date'].nunique(), esperado_dias[nombre])